# Word Segmentation Transformer Preset

This notebook is a customizable Transformer-style word-segmentation pipeline, adapted to the same approach as the provided reference notebook:

- Build character-level `B_WORD` / `I_WORD` / `E_WORD` labels from LST20.
- Fine-tune `AutoModelForTokenClassification`.
- Use overflow windows with stride for long training examples.
- Predict long test text with overlapping windows.
- Decode with either argmax or constrained Viterbi.
- Optionally ensemble multiple checkpoints.
- Write `Id,Predicted` in sample-submission order.

Preset archive facts:

- Test text: `ws_test.txt`
- Sample submission: `ws_sample_submission.csv`
- Label list: `ws_list.txt`
- Submission rows correspond to non-space characters only.

In [ ]:
# ============================================================
# 0. Configuration
# ============================================================
from pathlib import Path
import csv
import json
import zipfile
import ast
import re
import random
import unicodedata
from types import SimpleNamespace
from typing import Iterable, Iterator, List

import numpy as np
import pandas as pd

# ----- Competition files -----
ZIP_PATH = Path(r"word-segmentation.zip")
EXTRACTED_DIR = None  # Example: Path("/kaggle/input/super-ai-engineer-ss-6-word-segmentation")

TEST_TEXT_NAME = "ws_test.txt"
LABEL_LIST_NAME = "ws_list.txt"
SAMPLE_SUB_NAME = "ws_sample_submission.csv"
ID_COL = "Id"
SUB_TARGET_COL = "Predicted"

# ----- LST20 corpus -----
# Download/extract AIFORTHAI-LST20Corpus separately, then point this to the extracted folder.
LST20_PATH = None  # Example: Path(r"C:\Users\ZBook\Downloads\AIFORTHAI-LST20Corpus")
LST20_TRAIN_DIR = None  # Optional override, e.g. Path(..., "LST20_Corpus", "train")
LST20_EVAL_DIR = None   # Optional override, e.g. Path(..., "LST20_Corpus", "eval")
LST20_TXT_GLOB = "*.txt"

# Optional: if you already created JSONL files with {"chars": [...], "labels": [...]}.
TRAIN_JSONL = None
EVAL_JSONL = None
DATA_CACHE_DIR = Path("ws_data_cache")

# ----- Labels -----
LABELS = ["B_WORD", "I_WORD", "E_WORD"]
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}
ID_TO_LABEL = {idx: label for idx, label in enumerate(LABELS)}

# No S_WORD exists in this competition. The reference notebook uses B_WORD for singleton tokens.
SINGLE_CHAR_LABEL = "B_WORD"
SPACE_TOKEN = "_"

# ----- Model/training -----
BASE_MODEL = "airesearch/wangchanberta-base-att-spm-uncased"
TOKENIZER_NAME = BASE_MODEL
OUTPUT_DIR = Path("ws_transformer_model")
CACHE_DIR = None  # Example: Path(r"C:\Users\ZBook\.cache\huggingface")
LOCAL_FILES_ONLY = False
FORCE_CPU = False

MAX_LENGTH = 384
STRIDE = 96
OVERLAP_CHARS = 64
EPOCHS = 4.0
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
FP16 = True
GRADIENT_CHECKPOINTING = True
SAVE_STRATEGY = "epoch"  # "epoch", "steps", or "no"
SAVE_STEPS = 500
SAVE_TOTAL_LIMIT = 2
METRIC_FOR_BEST_MODEL = "f1"

# ----- Runtime controls -----
TRAIN_LIMIT = None       # Example: 100 for smoke tests
EVAL_LIMIT = None        # Example: 30 for smoke tests
REBUILD_JSONL = False
TRAIN_MODEL_NOW = False  # Set True after paths/deps are ready.

# Prediction controls.
MODEL_PATH = None        # If None after training, uses OUTPUT_DIR.
DECODE_MODE = "viterbi"  # "viterbi" or "argmax"
ENSEMBLE_SOURCES = []    # Example: [(r"path/to/checkpoint_a", 64, 1.0), (r"path/to/checkpoint_b", 128, 1.0)]
OUTPUT_CSV = Path("submission.csv")

assert ZIP_PATH.exists() or EXTRACTED_DIR is not None, "Set ZIP_PATH or EXTRACTED_DIR."

In [ ]:
# ============================================================
# 1. Load competition files
# ============================================================
def read_text_file(name):
    if EXTRACTED_DIR is None:
        with zipfile.ZipFile(ZIP_PATH) as zf:
            return zf.read(name).decode("utf-8-sig")
    return (Path(EXTRACTED_DIR) / name).read_text(encoding="utf-8-sig")


def read_csv_file(name):
    if EXTRACTED_DIR is None:
        with zipfile.ZipFile(ZIP_PATH) as zf:
            return pd.read_csv(zf.open(name))
    return pd.read_csv(Path(EXTRACTED_DIR) / name)


test_text = read_text_file(TEST_TEXT_NAME)
label_list_text = read_text_file(LABEL_LIST_NAME).strip()
sample_sub = read_csv_file(SAMPLE_SUB_NAME)

try:
    valid_labels = list(ast.literal_eval(label_list_text))
except Exception:
    valid_labels = [x.strip() for x in re.split(r"[\s,]+", label_list_text) if x.strip()]

print("text characters:", len(test_text))
print("spaces:", test_text.count(" "))
print("non-space characters:", sum(ch != " " for ch in test_text))
print("sample submission rows:", len(sample_sub))
print("valid labels:", valid_labels)
display(sample_sub.head(10))

assert set(LABELS).issubset(set(valid_labels)), "Configured LABELS must be in ws_list.txt"
assert len(sample_sub) == sum(ch != " " for ch in test_text), "Submission rows should match non-space characters."

In [ ]:
# ============================================================
# 2. Dependency check
# ============================================================
missing = []
try:
    import torch
except Exception:
    missing.append("torch")
try:
    from datasets import Dataset
except Exception:
    missing.append("datasets")
try:
    from transformers import (
        AutoModelForTokenClassification,
        AutoTokenizer,
        Trainer,
        TrainingArguments,
        default_data_collator,
    )
except Exception:
    missing.append("transformers")
try:
    from seqeval.metrics import f1_score as seqeval_f1_score
except Exception:
    missing.append("seqeval")

if missing:
    print("Missing optional training dependencies:", missing)
    print("Install them before training/prediction with this Transformer pipeline.")
else:
    print("Transformer training dependencies are available.")

In [ ]:
# ============================================================
# 3. LST20 parsing and JSONL preparation
# ============================================================
def token_to_bie_labels(token: str) -> List[str]:
    if len(token) == 0:
        return []
    if len(token) == 1:
        return [SINGLE_CHAR_LABEL]
    if len(token) == 2:
        return ["B_WORD", "E_WORD"]
    return ["B_WORD"] + ["I_WORD"] * (len(token) - 2) + ["E_WORD"]


def discover_lst20_split_dirs(root: Path):
    root = Path(root)
    candidates = [root, root / "LST20_Corpus", root / "lst20"]
    train_dir = LST20_TRAIN_DIR
    eval_dir = LST20_EVAL_DIR
    if train_dir is None:
        for base in candidates:
            for name in ["train", "training"]:
                p = base / name
                if p.exists():
                    train_dir = p
                    break
            if train_dir is not None:
                break
    if eval_dir is None:
        for base in candidates:
            for name in ["eval", "valid", "validation", "test"]:
                p = base / name
                if p.exists():
                    eval_dir = p
                    break
            if eval_dir is not None:
                break
    return Path(train_dir) if train_dir else None, Path(eval_dir) if eval_dir else None


def parse_lst20_doc(path: Path) -> tuple[str, List[str]]:
    raw_chars: List[str] = []
    labels: List[str] = []

    for line in path.read_text(encoding="utf-8-sig", errors="ignore").splitlines():
        line = line.strip()
        if not line:
            continue

        # LST20 lines are usually tab-separated: token POS NER ...
        token = line.split("\t")[0]
        if token == line:
            token = line.split()[0] if line.split() else ""

        if token == SPACE_TOKEN:
            raw_chars.append(" ")
            continue
        if not token:
            continue

        raw_chars.extend(token)
        labels.extend(token_to_bie_labels(token))

    return "".join(raw_chars), labels


def write_lst20_jsonl(split_dir: Path, output_path: Path, limit: int | None = None):
    paths = sorted(Path(split_dir).glob(LST20_TXT_GLOB))
    if limit is not None:
        paths = paths[:limit]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with output_path.open("w", encoding="utf-8") as f:
        for path in paths:
            text, labels = parse_lst20_doc(path)
            chars = [ch for ch in text if ch != " "]
            if len(chars) != len(labels):
                print("Skipping mismatched doc:", path, len(chars), len(labels))
                continue
            if not chars:
                continue
            f.write(json.dumps({"id": path.stem, "chars": chars, "labels": labels}, ensure_ascii=False) + "\n")
            count += 1
    print(f"wrote {count} docs -> {output_path}")
    return output_path


def prepare_jsonl_paths():
    global TRAIN_JSONL, EVAL_JSONL
    DATA_CACHE_DIR.mkdir(parents=True, exist_ok=True)

    if TRAIN_JSONL is not None and EVAL_JSONL is not None:
        return Path(TRAIN_JSONL), Path(EVAL_JSONL)

    if LST20_PATH is None:
        raise ValueError("Set LST20_PATH, or set TRAIN_JSONL and EVAL_JSONL.")

    train_dir, eval_dir = discover_lst20_split_dirs(Path(LST20_PATH))
    if train_dir is None or eval_dir is None:
        raise ValueError(f"Could not find LST20 train/eval directories under {LST20_PATH}. Set LST20_TRAIN_DIR and LST20_EVAL_DIR.")

    train_jsonl = DATA_CACHE_DIR / "train.jsonl"
    eval_jsonl = DATA_CACHE_DIR / "eval.jsonl"
    if REBUILD_JSONL or not train_jsonl.exists():
        write_lst20_jsonl(train_dir, train_jsonl, limit=TRAIN_LIMIT)
    if REBUILD_JSONL or not eval_jsonl.exists():
        write_lst20_jsonl(eval_dir, eval_jsonl, limit=EVAL_LIMIT)
    return train_jsonl, eval_jsonl

In [ ]:
# ============================================================
# 4. Dataset encoding
# ============================================================
def load_jsonl(path: Path, limit: int | None = None) -> Iterator[dict]:
    with Path(path).open(encoding="utf-8") as handle:
        for idx, line in enumerate(handle):
            if limit is not None and idx >= limit:
                break
            yield json.loads(line)


def encode_labeled_example(chars: List[str], labels: List[str], tokenizer, max_length: int, stride: int) -> List[dict]:
    encoded = tokenizer(
        chars,
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
    )

    rows = []
    for batch_index in range(len(encoded["input_ids"])):
        word_ids = encoded.word_ids(batch_index=batch_index)
        aligned_labels = []
        previous_word_id = None

        for word_id in word_ids:
            if word_id is None:
                aligned_labels.append(-100)
            elif word_id != previous_word_id:
                aligned_labels.append(LABEL_TO_ID[labels[word_id]])
            else:
                aligned_labels.append(-100)
            previous_word_id = word_id

        rows.append({
            "input_ids": encoded["input_ids"][batch_index],
            "attention_mask": encoded["attention_mask"][batch_index],
            "labels": aligned_labels,
        })
    return rows


def build_training_dataset(jsonl_path: Path, tokenizer, max_length: int, stride: int, cache_dir: Path | None = None, limit: int | None = None):
    from datasets import Dataset

    def generator() -> Iterable[dict]:
        for example in load_jsonl(jsonl_path, limit=limit):
            yield from encode_labeled_example(
                chars=example["chars"],
                labels=example["labels"],
                tokenizer=tokenizer,
                max_length=max_length,
                stride=stride,
            )

    return Dataset.from_generator(generator, cache_dir=str(cache_dir) if cache_dir else None)

In [ ]:
# ============================================================
# 5. Metrics, scoring, and decoding
# ============================================================
def compute_metrics(eval_prediction) -> dict:
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    mask = labels != -100
    token_acc = float((predictions[mask] == labels[mask]).mean()) if mask.any() else 0.0

    seq_predictions = []
    seq_labels = []
    for pred_row, label_row in zip(predictions, labels):
        cur_preds = []
        cur_labels = []
        for pred_id, label_id in zip(pred_row, label_row):
            if label_id == -100:
                continue
            cur_preds.append(ID_TO_LABEL[int(pred_id)])
            cur_labels.append(ID_TO_LABEL[int(label_id)])
        seq_predictions.append(cur_preds)
        seq_labels.append(cur_labels)

    return {
        "token_acc": token_acc,
        "f1": float(seqeval_f1_score(seq_labels, seq_predictions)),
    }


def fit_window_by_token_budget(chars: List[str], tokenizer, max_length: int) -> int:
    low, high = 1, len(chars)
    best = 1
    while low <= high:
        mid = (low + high) // 2
        encoded = tokenizer(chars[:mid], is_split_into_words=True, truncation=False)
        if len(encoded["input_ids"]) <= max_length:
            best = mid
            low = mid + 1
        else:
            high = mid - 1
    return best


def collect_char_scores(chars: List[str], tokenizer, model, max_length: int, overlap_chars: int, device: str) -> np.ndarray:
    import torch

    total_chars = len(chars)
    score_sums = np.zeros((total_chars, len(LABELS)), dtype=np.float32)
    hit_counts = np.zeros(total_chars, dtype=np.float32)

    model.eval()
    start = 0
    while start < total_chars:
        probe = chars[start : min(total_chars, start + max_length * 3)]
        window_len = fit_window_by_token_budget(probe, tokenizer, max_length=max_length)
        window_chars = chars[start : start + window_len]
        encoded = tokenizer(
            window_chars,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        word_ids = encoded.word_ids(batch_index=0)
        encoded = {key: value.to(device) for key, value in encoded.items()}

        with torch.no_grad():
            logits = model(**encoded).logits[0].detach().cpu().numpy()

        seen_word_ids = set()
        char_offset = 0
        for token_position, word_id in enumerate(word_ids):
            if word_id is None or word_id in seen_word_ids:
                continue
            out_idx = start + char_offset
            if out_idx >= total_chars:
                break
            score_sums[out_idx] += logits[token_position]
            hit_counts[out_idx] += 1
            char_offset += 1
            seen_word_ids.add(word_id)

        if start + window_len >= total_chars:
            break
        start += max(1, window_len - overlap_chars)

    missing = hit_counts == 0
    if missing.any():
        print("Warning: missing score positions:", int(missing.sum()))
    score_sums = score_sums / np.maximum(hit_counts[:, None], 1.0)
    return score_sums


def decode_viterbi(score_sums: np.ndarray) -> List[int]:
    neg = -1e18
    n = len(score_sums)
    dp = np.full((n, len(LABELS)), neg, dtype=np.float64)
    back = np.full((n, len(LABELS)), -1, dtype=np.int32)

    allowed_prev = {
        LABEL_TO_ID["B_WORD"]: [LABEL_TO_ID["B_WORD"], LABEL_TO_ID["I_WORD"], LABEL_TO_ID["E_WORD"]],
        LABEL_TO_ID["I_WORD"]: [LABEL_TO_ID["I_WORD"], LABEL_TO_ID["E_WORD"]],
        LABEL_TO_ID["E_WORD"]: [LABEL_TO_ID["B_WORD"]],
    }
    end_ok = {LABEL_TO_ID["B_WORD"], LABEL_TO_ID["E_WORD"]}

    dp[0, LABEL_TO_ID["B_WORD"]] = float(score_sums[0, LABEL_TO_ID["B_WORD"]])
    for idx in range(1, n):
        for cur in range(len(LABELS)):
            emit = float(score_sums[idx, cur])
            best_prev = -1
            best_score = neg
            for prev in range(len(LABELS)):
                if cur in allowed_prev.get(prev, []):
                    candidate = dp[idx - 1, prev] + emit
                    if candidate > best_score:
                        best_score = candidate
                        best_prev = prev
            dp[idx, cur] = best_score
            back[idx, cur] = best_prev

    best_last = max(end_ok, key=lambda state: dp[n - 1, state])
    output = [best_last]
    current = best_last
    for idx in range(n - 1, 0, -1):
        current = int(back[idx, current])
        output.append(current)
    output.reverse()
    return output


def predict_labels_from_scores(score_sums: np.ndarray, decode_mode: str) -> List[str]:
    if decode_mode == "argmax":
        prediction_ids = score_sums.argmax(axis=-1).tolist()
    elif decode_mode == "viterbi":
        prediction_ids = decode_viterbi(score_sums)
    else:
        raise ValueError(f"Unsupported decode mode: {decode_mode}")
    return [ID_TO_LABEL[int(pred_id)] for pred_id in prediction_ids]

In [ ]:
# ============================================================
# 6. Train
# ============================================================
def as_config(config=None):
    if config is None:
        return SimpleNamespace()
    if isinstance(config, dict):
        return SimpleNamespace(**config)
    return config


def train_model(config=None):
    config = as_config(config)
    import torch
    from transformers import AutoModelForTokenClassification, AutoTokenizer, Trainer, TrainingArguments, default_data_collator

    device = "cuda" if torch.cuda.is_available() and not getattr(config, "force_cpu", FORCE_CPU) else "cpu"
    tokenizer_source = getattr(config, "tokenizer_name", TOKENIZER_NAME) or getattr(config, "base_model", BASE_MODEL)
    tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_source,
        local_files_only=getattr(config, "local_files_only", LOCAL_FILES_ONLY),
        cache_dir=str(getattr(config, "cache_dir", CACHE_DIR)) if getattr(config, "cache_dir", CACHE_DIR) else None,
    )

    train_jsonl, eval_jsonl = prepare_jsonl_paths()
    dataset_cache_dir = DATA_CACHE_DIR / "hf_datasets"

    train_ds = build_training_dataset(
        jsonl_path=train_jsonl,
        tokenizer=tokenizer,
        max_length=getattr(config, "max_length", MAX_LENGTH),
        stride=getattr(config, "stride", STRIDE),
        cache_dir=dataset_cache_dir,
        limit=getattr(config, "train_limit", TRAIN_LIMIT),
    )
    eval_ds = build_training_dataset(
        jsonl_path=eval_jsonl,
        tokenizer=tokenizer,
        max_length=getattr(config, "max_length", MAX_LENGTH),
        stride=getattr(config, "stride", STRIDE),
        cache_dir=dataset_cache_dir,
        limit=getattr(config, "eval_limit", EVAL_LIMIT),
    )

    model = AutoModelForTokenClassification.from_pretrained(
        getattr(config, "base_model", BASE_MODEL),
        num_labels=len(LABELS),
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID,
        local_files_only=getattr(config, "local_files_only", LOCAL_FILES_ONLY),
        cache_dir=str(getattr(config, "cache_dir", CACHE_DIR)) if getattr(config, "cache_dir", CACHE_DIR) else None,
    )

    save_strategy = getattr(config, "save_strategy", SAVE_STRATEGY)
    save_steps = getattr(config, "save_steps", SAVE_STEPS)
    eval_strategy = save_strategy if save_strategy != "no" else "no"
    eval_steps = save_steps if save_strategy == "steps" else None

    training_args = TrainingArguments(
        output_dir=str(getattr(config, "output_dir", OUTPUT_DIR)),
        eval_strategy=eval_strategy,
        save_strategy=save_strategy,
        eval_steps=eval_steps,
        logging_strategy="steps",
        logging_steps=getattr(config, "logging_steps", 100),
        learning_rate=getattr(config, "learning_rate", LEARNING_RATE),
        per_device_train_batch_size=getattr(config, "train_batch_size", TRAIN_BATCH_SIZE),
        per_device_eval_batch_size=getattr(config, "eval_batch_size", EVAL_BATCH_SIZE),
        gradient_accumulation_steps=getattr(config, "gradient_accumulation_steps", GRADIENT_ACCUMULATION_STEPS),
        num_train_epochs=getattr(config, "epochs", EPOCHS),
        weight_decay=getattr(config, "weight_decay", WEIGHT_DECAY),
        warmup_ratio=getattr(config, "warmup_ratio", WARMUP_RATIO),
        save_total_limit=getattr(config, "save_total_limit", SAVE_TOTAL_LIMIT),
        save_steps=save_steps,
        load_best_model_at_end=save_strategy != "no",
        metric_for_best_model=getattr(config, "metric_for_best_model", METRIC_FOR_BEST_MODEL),
        greater_is_better=True,
        report_to="none",
        fp16=device == "cuda" and getattr(config, "fp16", FP16),
        gradient_checkpointing=getattr(config, "gradient_checkpointing", GRADIENT_CHECKPOINTING),
        dataloader_num_workers=getattr(config, "dataloader_num_workers", 0),
        remove_unused_columns=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        tokenizer=tokenizer,
        data_collator=default_data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train(resume_from_checkpoint=getattr(config, "resume_from_checkpoint", None))
    metrics = trainer.evaluate()
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
    trainer.save_model(str(getattr(config, "output_dir", OUTPUT_DIR)))
    tokenizer.save_pretrained(str(getattr(config, "output_dir", OUTPUT_DIR)))
    return metrics


if TRAIN_MODEL_NOW:
    train_model()
else:
    print("TRAIN_MODEL_NOW is False. Set paths/config, then call train_model({...}) manually.")

In [ ]:
# ============================================================
# 7. Prediction and submission
# ============================================================
def load_model_and_tokenizer(model_path: str | Path, tokenizer_name: str | Path | None = None, device: str = "cpu", local_files_only: bool = False, cache_dir: str | Path | None = None):
    from transformers import AutoModelForTokenClassification, AutoTokenizer

    tokenizer_source = tokenizer_name or model_path
    tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_source,
        local_files_only=local_files_only,
        cache_dir=str(cache_dir) if cache_dir else None,
    )
    model = AutoModelForTokenClassification.from_pretrained(
        model_path,
        cache_dir=str(cache_dir) if cache_dir else None,
        local_files_only=local_files_only,
    )
    model.to(device)
    return tokenizer, model


def predict_submission(config=None):
    config = as_config(config)
    import torch

    device = "cuda" if torch.cuda.is_available() and not getattr(config, "force_cpu", FORCE_CPU) else "cpu"
    model_path = getattr(config, "model_path", MODEL_PATH) or OUTPUT_DIR
    tokenizer_name = getattr(config, "tokenizer_name", TOKENIZER_NAME)

    tokenizer, model = load_model_and_tokenizer(
        model_path=model_path,
        tokenizer_name=tokenizer_name,
        device=device,
        local_files_only=getattr(config, "local_files_only", LOCAL_FILES_ONLY),
        cache_dir=getattr(config, "cache_dir", CACHE_DIR),
    )

    indexed_chars = [(idx + 1, ch) for idx, ch in enumerate(test_text) if ch != " "]
    char_ids = [idx for idx, _ in indexed_chars]
    chars = [ch for _, ch in indexed_chars]

    scores = collect_char_scores(
        chars=chars,
        tokenizer=tokenizer,
        model=model,
        max_length=getattr(config, "max_length", MAX_LENGTH),
        overlap_chars=getattr(config, "overlap_chars", OVERLAP_CHARS),
        device=device,
    )
    predicted = predict_labels_from_scores(scores, getattr(config, "decode_mode", DECODE_MODE))

    submission = sample_sub[[ID_COL]].copy()
    # The sample IDs are 1-based character ids for non-space chars in this competition.
    if list(submission[ID_COL].astype(int).values) != char_ids:
        print("Warning: sample IDs differ from computed non-space character positions. Keeping sample order.")
    submission[SUB_TARGET_COL] = predicted

    output_csv = Path(getattr(config, "output_csv", OUTPUT_CSV))
    submission.to_csv(output_csv, index=False)
    print(f"wrote {len(predicted)} rows to {output_csv}")
    display(submission.head(20))
    display(submission[SUB_TARGET_COL].value_counts().to_frame("predicted_count"))
    return output_csv


def predict_submission_ensemble(config=None):
    config = as_config(config)
    import torch

    sources = getattr(config, "sources", ENSEMBLE_SOURCES)
    if not sources:
        raise ValueError("Set ENSEMBLE_SOURCES or pass {'sources': [(model_path, overlap_chars, weight), ...]}.")

    device = "cuda" if torch.cuda.is_available() and not getattr(config, "force_cpu", FORCE_CPU) else "cpu"
    indexed_chars = [(idx + 1, ch) for idx, ch in enumerate(test_text) if ch != " "]
    char_ids = [idx for idx, _ in indexed_chars]
    chars = [ch for _, ch in indexed_chars]

    combined_scores = np.zeros((len(chars), len(LABELS)), dtype=np.float32)
    total_weight = 0.0

    for model_path, overlap_chars, weight in sources:
        print("loading", model_path, "overlap", overlap_chars, "weight", weight)
        tokenizer, model = load_model_and_tokenizer(
            model_path=model_path,
            tokenizer_name=getattr(config, "tokenizer_name", TOKENIZER_NAME),
            device=device,
            local_files_only=getattr(config, "local_files_only", LOCAL_FILES_ONLY),
            cache_dir=getattr(config, "cache_dir", CACHE_DIR),
        )
        scores = collect_char_scores(
            chars=chars,
            tokenizer=tokenizer,
            model=model,
            max_length=getattr(config, "max_length", MAX_LENGTH),
            overlap_chars=overlap_chars,
            device=device,
        )
        combined_scores += scores * float(weight)
        total_weight += float(weight)
        del model
        if device == "cuda":
            torch.cuda.empty_cache()

    combined_scores /= max(total_weight, 1e-12)
    predicted = predict_labels_from_scores(combined_scores, getattr(config, "decode_mode", DECODE_MODE))

    submission = sample_sub[[ID_COL]].copy()
    if list(submission[ID_COL].astype(int).values) != char_ids:
        print("Warning: sample IDs differ from computed non-space character positions. Keeping sample order.")
    submission[SUB_TARGET_COL] = predicted

    output_csv = Path(getattr(config, "output_csv", OUTPUT_CSV))
    submission.to_csv(output_csv, index=False)
    print(f"wrote {len(predicted)} rows to {output_csv}")
    display(submission.head(20))
    display(submission[SUB_TARGET_COL].value_counts().to_frame("predicted_count"))
    return output_csv

In [ ]:
# ============================================================
# 8. Optional LST20 eval-directory scoring
# ============================================================
def evaluate_on_lst20_docs(config=None):
    config = as_config(config)
    import torch
    from seqeval.metrics import f1_score as seqeval_f1_score

    device = "cuda" if torch.cuda.is_available() and not getattr(config, "force_cpu", FORCE_CPU) else "cpu"
    model_path = getattr(config, "model_path", MODEL_PATH) or OUTPUT_DIR
    tokenizer, model = load_model_and_tokenizer(
        model_path=model_path,
        tokenizer_name=getattr(config, "tokenizer_name", TOKENIZER_NAME),
        device=device,
        local_files_only=getattr(config, "local_files_only", LOCAL_FILES_ONLY),
        cache_dir=getattr(config, "cache_dir", CACHE_DIR),
    )

    eval_dir = getattr(config, "eval_dir", LST20_EVAL_DIR)
    if eval_dir is None:
        if LST20_PATH is None:
            raise ValueError("Set eval_dir or LST20_PATH.")
        _, eval_dir = discover_lst20_split_dirs(Path(LST20_PATH))

    paths = sorted(Path(eval_dir).glob(LST20_TXT_GLOB))
    if getattr(config, "limit", None) is not None:
        paths = paths[: config.limit]

    all_gold = []
    all_pred = []
    exact_docs = 0
    for path in paths:
        raw_text, gold_labels = parse_lst20_doc(path)
        chars = [ch for ch in raw_text if ch != " "]
        scores = collect_char_scores(
            chars=chars,
            tokenizer=tokenizer,
            model=model,
            max_length=getattr(config, "max_length", MAX_LENGTH),
            overlap_chars=getattr(config, "overlap_chars", OVERLAP_CHARS),
            device=device,
        )
        pred_labels = predict_labels_from_scores(scores, getattr(config, "decode_mode", DECODE_MODE))
        all_gold.append(gold_labels)
        all_pred.append(pred_labels)
        exact_docs += int(pred_labels == gold_labels)

    print("docs:", len(paths))
    print("exact docs:", exact_docs)
    print("seqeval f1:", seqeval_f1_score(all_gold, all_pred))

## Common Runs

Prepare LST20 and train:

```python
LST20_PATH = Path(r"C:\path\to\AIFORTHAI-LST20Corpus")
TRAIN_MODEL_NOW = False

train_model({
    "base_model": "airesearch/wangchanberta-base-att-spm-uncased",
    "tokenizer_name": "airesearch/wangchanberta-base-att-spm-uncased",
    "output_dir": "ws_model_wangchanberta",
    "max_length": 384,
    "stride": 96,
    "epochs": 4,
    "learning_rate": 2e-5,
    "train_batch_size": 2,
    "eval_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "metric_for_best_model": "f1",
    "gradient_checkpointing": True,
    "fp16": True,
})
```

Predict:

```python
predict_submission({
    "model_path": "ws_model_wangchanberta",
    "tokenizer_name": "airesearch/wangchanberta-base-att-spm-uncased",
    "decode_mode": "viterbi",
    "overlap_chars": 64,
    "output_csv": "submission.csv",
})
```

Ensemble:

```python
predict_submission_ensemble({
    "tokenizer_name": "airesearch/wangchanberta-base-att-spm-uncased",
    "decode_mode": "viterbi",
    "output_csv": "submission_ensemble.csv",
    "sources": [
        ("checkpoint_a", 64, 1.0),
        ("checkpoint_b", 128, 1.0),
    ],
})
```

Customization knobs that matter most:

- `BASE_MODEL` / `TOKENIZER_NAME`
- `MAX_LENGTH`, `STRIDE`, `OVERLAP_CHARS`
- `SINGLE_CHAR_LABEL`
- `DECODE_MODE`
- `TRAIN_LIMIT`, `EVAL_LIMIT` for smoke tests
- `LOCAL_FILES_ONLY` and `CACHE_DIR` for offline runs